## Основной вариант

In [6]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = "1"

import re
import csv
import sys
import time
import json
import math
import pickle
import pygame
import numpy as np
import onnxruntime as ort
from collections import deque, Counter
from numba import jit

# ------------------------- ПУТИ -------------------------
CLF = "xgb"
REG = "lgb"

ONNX_CLF = f"DATA/onnx_models/clf_{CLF}.onnx"
ONNX_REG = f"DATA/onnx_models/reg_{REG}.onnx"
REG_MODEL_PATH = f"DATA/real_time/hybrid_pipeline_{REG}.pkl"
FEATURES_PATH = "DATA/real_time/selected_50_features.json"
FOLDER_NAME = "DATA/test_realtime_system"

WIDTH, HEIGHT = 1000, 600

CLASS_LABELS = {
    0: "НОРМА (Винты целы)",
    1: "ЛЕГКОЕ ПОВРЕЖДЕНИЕ: 1 винт (0.5 см)",
    2: "СРЕДНЕЕ ПОВРЕЖДЕНИЕ: 1 винт (1.0 см)",
    3: "ТЯЖЕЛОЕ ПОВРЕЖДЕНИЕ: 1 винт (1.5 см)",
    4: "СРЕДНЕЕ ПОВРЕЖДЕНИЕ: 2 винта (0.5 см)",
    5: "ТЯЖЕЛОЕ ПОВРЕЖДЕНИЕ: 3 винта (0.5 см)"
}

CLASS_COLORS = [(0, 255, 100), (255, 255, 0), (255, 170, 0),
                (255, 0, 50), (255, 170, 0), (255, 0, 50)]

# -------------------- ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ --------------------
@jit(nopython=True, cache=True)
def extract_features(window, out):
    for axis in range(6):
        x = window[:, axis]
        n = len(x)

        # RAW
        min_val = max_val = sum_val = x[0]
        sum_abs = abs(x[0])
        for j in range(1, n):
            val = x[j]
            if val < min_val:
                min_val = val
            if val > max_val:
                max_val = val
            sum_val += val
            sum_abs += abs(val)

        mean_val = sum_val / n
        ptp_val = max_val - min_val

        base = axis * 18
        out[base:base+6] = [min_val, max_val, mean_val, ptp_val, sum_abs, sum_val]

        # 1-ый DIFF
        if n > 1:
            d0 = x[1] - x[0]
            min_val = max_val = sum_val = d0
            sum_abs = abs(d0)
            for j in range(1, n-1):
                d = x[j+1] - x[j]
                if d < min_val:
                    min_val = d
                if d > max_val:
                    max_val = d
                sum_val += d
                sum_abs += abs(d)
            out[base+6:base+12] = [min_val, max_val, sum_val/(n-1), max_val-min_val, sum_abs, sum_val]
        else:
            out[base+6:base+12] = [0.0] * 6

        # 2-ой DIFF
        if n > 2:
            d2_0 = x[2] - 2*x[1] + x[0]
            min_val = max_val = sum_val = d2_0
            sum_abs = abs(d2_0)
            for j in range(2, n-1):
                d2 = x[j+1] - 2*x[j] + x[j-1]
                if d2 < min_val:
                    min_val = d2
                if d2 > max_val:
                    max_val = d2
                sum_val += d2
                sum_abs += abs(d2)
            out[base+12:base+18] = [min_val, max_val, sum_val/(n-2), max_val-min_val, sum_abs, sum_val]
        else:
            out[base+12:base+18] = [0.0] * 6

# ------------------------- ЗАГРУЗКА -------------------------
required = [ONNX_CLF, ONNX_REG, REG_MODEL_PATH, FEATURES_PATH]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print(f"Отсутствуют: {missing}")
    sys.exit(1)

try:
    opts = ort.SessionOptions()
    opts.enable_cpu_mem_arena = True
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    opts.intra_op_num_threads = 1
    opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

    sess_clf = ort.InferenceSession(ONNX_CLF, opts, providers=["CPUExecutionProvider"])
    sess_reg = ort.InferenceSession(ONNX_REG, opts, providers=["CPUExecutionProvider"])

    with open(REG_MODEL_PATH, "rb") as f:
        hybrid = pickle.load(f)
    with open(FEATURES_PATH) as f:
        selected = json.load(f)

except Exception as e:
    print(f"{e}")
    sys.exit(1)

# ------------------------- ДАННЫЕ -------------------------
ordinal_models = hybrid["ordinal_models"]
scaler = hybrid["scaler"]
l_min, l_max = hybrid["latent_min"], hybrid["latent_max"]
r_min, r_max = hybrid["reg_min"], hybrid["reg_max"]
alpha = hybrid["alpha"]

one_minus_alpha = 1 - alpha

W = np.vstack([m.coef_.ravel() for m in ordinal_models]).astype(np.float32).T
b = np.array([m.intercept_[0] for m in ordinal_models], dtype=np.float32)
mean = scaler.mean_.astype(np.float32)
scale = scaler.scale_.astype(np.float32)
inv_scale = 1.0 / scale

denom_l = l_max - l_min
denom_r = r_max - r_min

# -------------------- МАППИНГ ПРИЗНАКОВ --------------------
feature_indices = []
for feat in selected:
    prefix, stat = feat.rsplit("_", 1)
    axis = {'gx': 0, 'gy': 1, 'gz': 2, 'ax': 3, 'ay': 4, 'az': 5}[prefix.split('_')[0]]
    stat_idx = {"Min": 0, "Max": 1, "Mean": 2, "Ptp": 3, "L1": 4, "CumsumFinal": 5}[stat]
    sig_type = 2 if "dif_dif" in prefix else (1 if "dif" in prefix else 0)
    idx = axis * 18 + sig_type * 6 + stat_idx
    feature_indices.append(idx)

feature_indices = np.array(feature_indices, dtype=np.int32)

# ------------------------- CSV -------------------------
def get_file_path():
    os.makedirs(FOLDER_NAME, exist_ok=True)
    nums = [int(re.search(r'data_(\d+)\.csv', f).group(1))
            for f in os.listdir(FOLDER_NAME) if re.search(r'data_(\d+)\.csv', f)]
    next_num = max(nums) + 1 if nums else 1
    return os.path.join(FOLDER_NAME, f'simulated_imu_data_{next_num}.csv')

# ----------------------- СИМУЛЯТОР -----------------------
class MockMsg:
    __slots__ = ('m_type', 'time_usec', 'id', 'xacc', 'yacc', 'zacc', 'xgyro', 'ygyro', 'zgyro')
    def __init__(self, m_type):
        self.m_type = m_type
        self.time_usec = int(time.time() * 1e6)
        self.id = 1
        cycle = (time.time() % 20) / 20.0
        noise = 1.0 + (10.0 * math.sin(cycle * math.pi)) if cycle > 0.5 else 1.0
        if m_type == 'HIGHRES_IMU':
            self.xacc = np.random.normal(0, 0.5 * noise)
            self.yacc = np.random.normal(0, 0.5 * noise)
            self.zacc = np.random.normal(-9.8, 0.5 * noise)
            self.xgyro = np.random.normal(0, 0.1 * noise)
            self.ygyro = np.random.normal(0, 0.1 * noise)
            self.zgyro = np.random.normal(0, 0.1 * noise)
    def get_type(self):
        return self.m_type

# ----------------------- НАСТРОЙКИ -----------------------
WINDOW_SIZE = 32
STRIDE = 8
CSV_BATCH_SIZE = 100
HISTORY_LEN = 250
FPS = 60
SIMULATION_HZ = 50.0

# -------------------- ИНИЦИАЛИЗАЦИЯ --------------------
file_path = get_file_path()
csv_file = open(file_path, 'w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['time_usec', 'ax', 'ay', 'az', 'gx', 'gy', 'gz',
                 'mx', 'my', 'mz', 'abs_pressure', 'diff_pressure',
                 'pressure_alt', 'temperature', 'fields_updated', 'id'])

imu_buffer = np.zeros((WINDOW_SIZE, 6), dtype=np.float32)
X_raw = np.empty(108, dtype=np.float32)
X_sel = np.empty((1, 50), dtype=np.float32)
X_scaled = np.empty((1, 50), dtype=np.float32)

latency_history = deque([0.0] * HISTORY_LEN, maxlen=HISTORY_LEN)
risk_history = deque([0.0] * HISTORY_LEN, maxlen=HISTORY_LEN)
csv_buf = deque()

current_risk = 0.0
current_class = 0
smoothed_class = 0
current_latency = 0.0
avg_latency = 0.0
packets = 0
pending = 0
total_pred = 0

total_feature_time = 0.0
total_classifier_time = 0.0
total_scaler_time = 0.0
total_ordinal_time = 0.0
total_regression_time = 0.0
total_fusion_time = 0.0
total_all_time = 0.0

# --------------- ПОЛЗУНКИ ---------------
threshold_values = [i*0.005 for i in range(200)]
threshold_idx = 0  
threshold = threshold_values[threshold_idx]

window_count_values = [1, 3, 5, 7, 9]
window_count_idx = 0  
window_count = window_count_values[window_count_idx]

overlap_values = [0.0, 0.25, 0.5, 0.75, 0.875]
overlap_idx = 4
overlap = overlap_values[overlap_idx]
STRIDE = max(1, int(WINDOW_SIZE * (1 - overlap)))

prediction_buffer = deque(maxlen=9)

# --------------------- ФУНКЦИЯ ГОЛОСОВАНИЯ ---------------------
def majority_vote(buffer, k):
    if k <= 0 or len(buffer) == 0:
        return -1
    last_k = list(buffer)[-k:]
    valid = [x for x in last_k if x != -1]
    if not valid: return -1
    
    cnt = Counter(valid)
    max_freq = max(cnt.values())
    most_common = [cls for cls, freq in cnt.items() if freq == max_freq]
    return most_common[0] if len(most_common) == 1 else int(round(sum(most_common) / len(most_common)))
 

# --------------------- КЛАСС ПОЛЗУНКА ---------------------
class Slider:
    def __init__(self, x, y, w, h, values, initial_idx, label):
        self.rect = pygame.Rect(x, y, w, h)
        self.values = values
        self.idx = initial_idx
        self.label = label
        self.dragging = False

    def get_value(self):
        return self.values[self.idx]

    def draw(self, screen, font):
        label_surf = font.render(self.label, True, (220, 220, 220))
        label_x = self.rect.x + self.rect.width//2 - label_surf.get_width()//2
        screen.blit(label_surf, (label_x, self.rect.y - 25))
        
        pygame.draw.rect(screen, (120, 120, 140), self.rect, 2, border_radius=4)
        
        knob_x = self.rect.x + (self.idx / (len(self.values)-1)) * self.rect.width
        knob_rect = pygame.Rect(knob_x - 5, self.rect.y - 3, 10, self.rect.height + 6)
        pygame.draw.rect(screen, (232, 207, 122), knob_rect, border_radius=8)
        
        val_str = str(self.get_value()) if not isinstance(self.get_value(), float) else f"{self.get_value()*100:.1f}%"
        val_surf = font.render(val_str, True, (200, 200, 200))
        val_x = self.rect.x + self.rect.width//2 - val_surf.get_width()//2
        screen.blit(val_surf, (val_x, self.rect.bottom + 5))

    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.rect.collidepoint(event.pos):
                self.dragging = True
                self._set_from_pos(event.pos[0])
        elif event.type == pygame.MOUSEBUTTONUP and event.button == 1:
            self.dragging = False
        elif event.type == pygame.MOUSEMOTION and self.dragging:
            self._set_from_pos(event.pos[0])

    def _set_from_pos(self, mouse_x):
        ratio = (mouse_x - self.rect.x) / self.rect.width
        ratio = max(0.0, min(1.0, ratio))
        self.idx = int(round(ratio * (len(self.values)-1)))
        self.idx = max(0, min(len(self.values)-1, self.idx))

# ------------------------- PYGAME -------------------------
pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("REAL-TIME DETECTION")
clock = pygame.time.Clock()

font_l = pygame.font.SysFont("verdana", 28, bold=True)
font_m = pygame.font.SysFont("verdana", 22, bold=True)
font_ms = pygame.font.SysFont("verdana", 20, bold=True)
font_s = pygame.font.SysFont("verdana", 14)

last_time = time.perf_counter()
interval = 1.0 / SIMULATION_HZ

lower_risk_limit = 15
upper_risk_limit = 30

# # --------------- СОЗДАНИЕ ПОЛЗУНКОВ ---------------
slider_y = 160
slider_height = 18
slider_width = 150

slider_overlap = Slider(20, slider_y, slider_width, slider_height,
                        overlap_values, overlap_idx, "Наложение")
slider_window = Slider(68 + slider_width, slider_y, slider_width, slider_height,
                       window_count_values, window_count_idx, "Количество окон")
slider_thresh = Slider(116 + 2*slider_width, slider_y, slider_width, slider_height,
                       threshold_values, threshold_idx, "Порог уверенности")

# --------------------- ОСНОВНОЙ ЦИКЛ ---------------------
lst_reg = []
lst_cl = []

running = True
try:
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
    
            slider_overlap.handle_event(event)
            slider_window.handle_event(event)
            slider_thresh.handle_event(event)
    
        new_overlap = slider_overlap.get_value()
        if new_overlap != overlap:
            overlap = new_overlap
            STRIDE = max(1, int(WINDOW_SIZE * (1 - overlap)))
    
        new_window_count = slider_window.get_value()
        if new_window_count != window_count:
            window_count = new_window_count
    
        new_threshold = slider_thresh.get_value()
        if new_threshold != threshold:
            threshold = new_threshold
    
        now = time.perf_counter()
    
        while (now - last_time) >= interval and running:
            msg = MockMsg('HIGHRES_IMU')
    
            csv_buf.append([msg.time_usec, msg.xacc, msg.yacc, msg.zacc,
                           msg.xgyro, msg.ygyro, msg.zgyro, 0, 0, 0, 0, 0, 0, 25, 0, packets, '', '', '', ''])
    
            imu_buffer[:-1] = imu_buffer[1:]
            imu_buffer[-1] = [msg.xgyro, msg.ygyro, msg.zgyro, msg.xacc, msg.yacc, msg.zacc]
    
            packets += 1
            pending += 1
    
            if packets >= WINDOW_SIZE and pending >= STRIDE:
    
                # 1. Features extraction
                t_start = time.perf_counter()
                extract_features(imu_buffer, X_raw)
                X_sel[0, :] = X_raw[feature_indices]
                t_feature = time.perf_counter() - t_start
    
                # 2. Classifier inference
                t0 = time.perf_counter()
                probs = sess_clf.run(["probabilities"], {"input": X_sel})[0][0]
                max_prob = np.max(probs)
                raw_class = int(np.argmax(probs))
                current_class = raw_class if max_prob >= threshold else -1
                prediction_buffer.append(current_class)
                smoothed_class = majority_vote(prediction_buffer, window_count)
                # smoothed_class = int(sess_clf.run(None, {"input": X_sel})[0][0])
                t_classifier = time.perf_counter() - t0
                
                # 3. Scaling
                t0 = time.perf_counter()
                np.subtract(X_sel, mean, out=X_scaled)
                np.multiply(X_scaled, inv_scale, out=X_scaled)
                t_scaler = time.perf_counter() - t0
    
                # 4. Ordinal inference
                t0 = time.perf_counter()
                ordinal = (np.dot(X_scaled, W) + b).mean()
                ord_n = (ordinal - l_min) / denom_l
                t_ordinal = time.perf_counter() - t0
    
                # 5. Regression inference
                t0 = time.perf_counter()
                reg = sess_reg.run(None, {"input": X_sel})[0][0][0]
                reg_n = (reg - r_min) / denom_r
                t_regression = time.perf_counter() - t0
    
                # 6. Fusion
                t0 = time.perf_counter()
                current_risk = min(max(0, 100 * (alpha * ord_n + one_minus_alpha * reg_n)), 100)
                t_end = time.perf_counter()
                t_fusion = t_end - t0
    
                if total_pred > 1:
                    total_feature_time += t_feature
                    total_classifier_time += t_classifier
                    total_scaler_time += t_scaler
                    total_ordinal_time += t_ordinal
                    total_regression_time += t_regression
                    total_fusion_time += t_fusion
                    total_all_time += (t_end - t_start)
                    current_latency = (t_end - t_start) * 1000
                    
                lst_reg.append(t_feature+t_scaler+t_ordinal)
                lst_cl.append(current_latency)
                total_pred += 1

                if total_pred > 1000:
                    running = False
    
                risk_history.append(float(current_risk))
                latency_history.append(float(current_latency))
    
                pending = 0
    
            last_time += interval
    
        if len(csv_buf) >= CSV_BATCH_SIZE:
            writer.writerows(csv_buf)
            csv_buf.clear()
    
        screen.fill((15, 15, 25))
        display_class = smoothed_class
        
        if display_class == -1:
            color = (150, 150, 150)
            class_text = "МОДЕЛЬ НЕ УВЕРЕНА В ПРЕДСКАЗАНИИ"
        else:
            color = CLASS_COLORS[display_class] if 0 <= display_class <= 5 else (200, 200, 200)
            class_text = CLASS_LABELS.get(display_class, f"КЛАСС {display_class}")
    
        pygame.draw.rect(screen, (40, 40, 60), (20, 20, 545, 100), border_radius=10)
        pygame.draw.rect(screen, color, (20, 20, 545, 100), width=3, border_radius=10)
        screen.blit(font_s.render("ТЕКУЩЕЕ СОСТОЯНИЕ:", True, (220, 220, 220)), (40, 35))
        screen.blit(font_ms.render(class_text, True, color), (40, 65))
    
        pygame.draw.rect(screen, (40, 40, 60), (WIDTH-320, 20, 300, 100), border_radius=10)
        screen.blit(font_s.render("ЗАДЕРЖКА", True, (220, 220, 220)), (WIDTH - 300, 40))
        screen.blit(font_m.render(f"{current_latency:.3f} ms", True, (220, 220, 220)), (WIDTH - 300, 65))
    
        rect = pygame.Rect(WIDTH-160, 35, 120, 70)
        pygame.draw.rect(screen, (20, 20, 30), rect, border_radius=5)
        max_lat = max(max(latency_history), 0.8)
        lat_list = list(latency_history)
    
        for i in range(1, len(lat_list)):
            x1 = rect.x + int((i - 1) * rect.width / HISTORY_LEN)
            y1 = rect.bottom - int((lat_list[i - 1]/max_lat) * rect.height)
            x2 = rect.x + int(i * rect.width / HISTORY_LEN)
            y2 = rect.bottom - int((lat_list[i] / max_lat) * rect.height)
            pygame.draw.line(screen, (100, 200, 255), (x1, y1), (x2, y2), 2)
    
        screen.blit(font_s.render(f"Файл сессии: {os.path.basename(file_path)}", True, (200, 200, 200)), (WIDTH - 318, 128))
        screen.blit(font_s.render(f"Сохранено отсчётов: {packets:,}", True, (120, 210, 245)), (WIDTH - 318, 176))  # небесный
        screen.blit(font_s.render(f"Выполнено предсказаний: {total_pred:,}", True, (255, 215, 120)), (WIDTH - 318, 152))  # шампань
    
        slider_overlap.draw(screen, font_s)
        slider_window.draw(screen, font_s)
        slider_thresh.draw(screen, font_s)
    
        g_rect = pygame.Rect(20, 220, WIDTH - 40, HEIGHT - 240)
        pygame.draw.rect(screen, (20, 20, 30), g_rect, border_radius=5)
    
        for p, col in [(lower_risk_limit, (0, 200, 100)), (upper_risk_limit, (255, 170, 0))]:
            y = g_rect.bottom - int((p / 100) * g_rect.height)
            pygame.draw.line(screen, col, (g_rect.left, y), (g_rect.right, y), 1)
            screen.blit(font_s.render(f"{p}%", True, col), (g_rect.left + 5, y - 20))
    
        if len(risk_history) > 1:
            risk_list = list(risk_history)
            pts = [(g_rect.left + int(i * g_rect.width / (HISTORY_LEN - 1)),
                    g_rect.bottom - int((v / 100) * g_rect.height)) for i, v in enumerate(risk_list)]
            s_color = (0, 200, 100) if current_risk < lower_risk_limit else ((255, 170, 0) if current_risk < upper_risk_limit else (255, 50, 50))
            if len(pts) > 1:
                pygame.draw.lines(screen, s_color, False, pts, 3)
    
        screen.blit(font_l.render(f"Риск: {current_risk:.1f}%", True, s_color), (g_rect.left + 20, g_rect.top + 20))
    
        pygame.display.flip()
        clock.tick(FPS)
        
finally:
    if csv_buf:
        writer.writerows(csv_buf)
    csv_file.close()
    pygame.quit()
    
    print("="*48)
    print(" "*14, "ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ", sep="")
    print("="*48)
    
    label_width = 34
    
    if total_pred > 0:
        avg_feature = (total_feature_time / total_pred) * 1000
        avg_classifier = (total_classifier_time / total_pred) * 1000
        avg_scaler = (total_scaler_time / total_pred) * 1000
        avg_ordinal = (total_ordinal_time / total_pred) * 1000
        avg_regression = (total_regression_time / total_pred) * 1000
        avg_fusion = (total_fusion_time / total_pred) * 1000
    
        avg_risk = avg_scaler + avg_ordinal + avg_regression + avg_fusion
        avg_all = avg_feature + avg_classifier + avg_risk
    
        print(f"\n{'Суммарная задержка обработки:':<{label_width}}{avg_all:.3f} ms\n")
        print(f"{'Суммарная задержка оценки риска:':<{label_width}}{avg_risk:.3f} ms\n")
        print(f"{'Задержка извлечения признаков:':<{label_width}}{avg_feature:.3f} ms")
        print(f"{'Задержка классификации:':<{label_width}}{avg_classifier:.3f} ms")
        print(f"{'Задержка масштабирования:':<{label_width}}{avg_scaler:.3f} ms")
        print(f"{'Задержка порядковой модели:':<{label_width}}{avg_ordinal:.3f} ms")
        print(f"{'Задержка регрессионной модели:':<{label_width}}{avg_regression:.3f} ms")
    
        print(f"\n{'Количество предсказаний:':<{label_width}}{total_pred}")
        print(f"{'Пропускная способность:':<{label_width}}{1000/avg_all:.1f} пред/с")
    else:
        print("\nНет предсказаний")
    
    print("\n", "="*48, sep="")


              ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ

Суммарная задержка обработки:     0.364 ms

Суммарная задержка оценки риска:  0.118 ms

Задержка извлечения признаков:    0.024 ms
Задержка классификации:           0.222 ms
Задержка масштабирования:         0.010 ms
Задержка порядковой модели:       0.035 ms
Задержка регрессионной модели:    0.070 ms

Количество предсказаний:          587
Пропускная способность:           2745.5 пред/с



In [4]:
p50 = round(np.percentile(lst_reg, 50) * 1000, 3)
p75 = round(np.percentile(lst_reg, 75) * 1000, 3)
p95 = round(np.percentile(lst_reg, 95) * 1000, 3)
p98 = round(np.percentile(lst_reg, 98) * 1000, 3)
p99 = round(np.percentile(lst_reg, 99) * 1000, 3)

print(p50, p75, p95, p98, p99)

p50 = round(np.percentile(lst_cl, 50), 3)
p75 = round(np.percentile(lst_cl, 75), 3)
p95 = round(np.percentile(lst_cl, 95), 3)
p98 = round(np.percentile(lst_cl, 98), 3)
p99 = round(np.percentile(lst_cl, 99), 3)

print(p50, p75, p95, p98, p99)



0.068 0.082 0.101 0.108 0.116
0.353 0.439 0.533 0.557 0.584


## Без окон и уверенности в себе

In [5]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = "1"

import re
import csv
import sys
import time
import json
import math
import pickle
import pygame
import numpy as np
import onnxruntime as ort
from collections import deque
from numba import jit

# ПУТИ

CLF = "xgb"
REG = "lgb"

ONNX_CLF = f"clf_{CLF}.onnx"
ONNX_REG = f"reg_{REG}.onnx"
REG_MODEL_PATH = f"DATA/real_time/hybrid_pipeline_{REG}.pkl"      
FEATURES_PATH = "DATA/real_time/selected_50_features.json"  
FOLDER_NAME = "DATA/test_realtime_system"    


WIDTH, HEIGHT = 1000, 600

CLASS_LABELS = { 
    0: "НОРМА (Винты целы)", 
    1: "ЛЕГКОЕ ПОВРЕЖДЕНИЕ: 1 винт (0.5 см)",
    2: "СРЕДНЕЕ ПОВРЕЖДЕНИЕ: 1 винт (1.0 см)",
    3: "ТЯЖЕЛОЕ ПОВРЕЖДЕНИЕ: 1 винт (1.5 см)",
    4: "СРЕДНЕЕ ПОВРЕЖДЕНИЕ: 2 винта (0.5 см)",
    5: "ТЯЖЕЛОЕ ПОВРЕЖДЕНИЕ: 3 винта (0.5 см)"
}

CLASS_COLORS = [(0, 255, 100), (255, 255, 0), (255, 170, 0), 
                (255, 0, 50), (255, 170, 0), (255, 0, 50)]


# ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ

@jit(nopython=True, cache=True)
def extract_features(window, out):
    for axis in range(6):
        x = window[:, axis]
        n = len(x)
        
        # RAW
        min_val = max_val = sum_val = x[0]
        sum_abs = abs(x[0])
        for j in range(1, n):
            val = x[j]
            if val < min_val:
                min_val = val
            if val > max_val:
                max_val = val
            sum_val += val
            sum_abs += abs(val)
        
        mean_val = sum_val / n
        ptp_val = max_val - min_val
        
        base = axis * 18
        out[base:base+6] = [min_val, max_val, mean_val, ptp_val, sum_abs, sum_val]
        
        # 1-ый DIFF
        if n > 1:
            d0 = x[1] - x[0]
            min_val = max_val = sum_val = d0
            sum_abs = abs(d0)
            for j in range(1, n-1):
                d = x[j+1] - x[j]
                if d < min_val:
                    min_val = d
                if d > max_val:
                    max_val = d
                sum_val += d
                sum_abs += abs(d)
            out[base+6:base+12] = [min_val, max_val, sum_val/(n-1), max_val-min_val, sum_abs, sum_val]
        else:
            out[base+6:base+12] = [0.0] * 6
        
        # 2-ой DIFF
        if n > 2:
            d2_0 = x[2] - 2*x[1] + x[0]
            min_val = max_val = sum_val = d2_0
            sum_abs = abs(d2_0)
            for j in range(2, n-1):
                d2 = x[j+1] - 2*x[j] + x[j-1]
                if d2 < min_val:
                    min_val = d2
                if d2 > max_val:
                    max_val = d2
                sum_val += d2
                sum_abs += abs(d2)
            out[base+12:base+18] = [min_val, max_val, sum_val/(n-2), max_val-min_val, sum_abs, sum_val]
        else:
            out[base+12:base+18] = [0.0] * 6


# ЗАГРУЗКА

required = [ONNX_CLF, ONNX_REG, REG_MODEL_PATH, FEATURES_PATH]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print(f"Отсутствуют: {missing}")
    sys.exit(1)

try:
    # ONNX Runtime с оптимизациями
    opts = ort.SessionOptions()
    opts.enable_cpu_mem_arena = True
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    opts.intra_op_num_threads = 1
    opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    
    sess_clf = ort.InferenceSession(ONNX_CLF, opts, providers=["CPUExecutionProvider"])
    sess_reg = ort.InferenceSession(ONNX_REG, opts, providers=["CPUExecutionProvider"])

    with open(REG_MODEL_PATH, "rb") as f:    
        hybrid = pickle.load(f)
    with open(FEATURES_PATH) as f:           
        selected = json.load(f)
    
except Exception as e:
    print(f"{e}")
    sys.exit(1)

# ДАННЫЕ

ordinal_models = hybrid["ordinal_models"]
scaler = hybrid["scaler"]
l_min, l_max = hybrid["latent_min"], hybrid["latent_max"]
r_min, r_max = hybrid["reg_min"], hybrid["reg_max"]
alpha = hybrid["alpha"]

one_minus_alpha = 1 - alpha

W = np.vstack([m.coef_.ravel() for m in ordinal_models]).astype(np.float32).T
b = np.array([m.intercept_[0] for m in ordinal_models], dtype=np.float32)
mean = scaler.mean_.astype(np.float32)
scale = scaler.scale_.astype(np.float32)
inv_scale = 1.0 / scale

denom_l = l_max - l_min
denom_r = r_max - r_min

# МАППИНГ ПРИЗНАКОВ

feature_indices = []
for feat in selected:
    prefix, stat = feat.rsplit("_", 1)
    axis = {'gx': 0, 'gy': 1, 'gz': 2, 'ax': 3, 'ay': 4, 'az': 5}[prefix.split('_')[0]]
    stat_idx = {"Min": 0, "Max": 1, "Mean": 2, "Ptp": 3, "L1": 4, "CumsumFinal": 5}[stat]
    sig_type = 2 if "dif_dif" in prefix else (1 if "dif" in prefix else 0)
    idx = axis * 18 + sig_type * 6 + stat_idx
    feature_indices.append(idx)

feature_indices = np.array(feature_indices, dtype=np.int32)

# CSV

def get_file_path():
    os.makedirs(FOLDER_NAME, exist_ok=True)
    nums = [int(re.search(r'data_(\d+)\.csv', f).group(1)) 
            for f in os.listdir(FOLDER_NAME) if re.search(r'data_(\d+)\.csv', f)]
    next_num = max(nums) + 1 if nums else 1
    return os.path.join(FOLDER_NAME, f'simulated_imu_data_{next_num}.csv')

# СИМУЛЯТОР

class MockMsg:
    __slots__ = ('m_type', 'time_usec', 'id', 'xacc', 'yacc', 'zacc', 'xgyro', 'ygyro', 'zgyro')
    def __init__(self, m_type):
        self.m_type = m_type
        self.time_usec = int(time.time() * 1e6)
        self.id = 1
        cycle = (time.time() % 20) / 20.0
        noise = 1.0 + (10.0 * math.sin(cycle * math.pi)) if cycle > 0.5 else 1.0
        if m_type == 'HIGHRES_IMU':
            self.xacc = np.random.normal(0, 0.5 * noise)
            self.yacc = np.random.normal(0, 0.5 * noise)
            self.zacc = np.random.normal(-9.8, 0.5 * noise)
            self.xgyro = np.random.normal(0, 0.1 * noise)
            self.ygyro = np.random.normal(0, 0.1 * noise)
            self.zgyro = np.random.normal(0, 0.1 * noise)
    def get_type(self):
        return self.m_type

# НАСТРОЙКИ 

WINDOW_SIZE = 32
STRIDE = 8
CSV_BATCH_SIZE = 100
HISTORY_LEN = 150
FPS = 60
SIMULATION_HZ = 50.0

# ИНИЦИАЛИЗАЦИЯ 

file_path = get_file_path()
csv_file = open(file_path, 'w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['time_usec', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 
                 'mx', 'my', 'mz', 'abs_pressure', 'diff_pressure', 
                 'pressure_alt', 'temperature', 'fields_updated', 'id'])

# ПРЕДВЫДЕЛЕНИЕ

imu_buffer = np.zeros((WINDOW_SIZE, 6), dtype=np.float32)
X_raw = np.empty(108, dtype=np.float32)
X_sel = np.empty((1, 50), dtype=np.float32)
X_scaled = np.empty((1, 50), dtype=np.float32)

latency_history = deque([0.0] * HISTORY_LEN, maxlen=HISTORY_LEN)
risk_history = deque([0.0] * HISTORY_LEN, maxlen=HISTORY_LEN)
csv_buf = deque()

current_risk = 0.0
current_class = 0
current_latency = 0.0
avg_latency = 0.0
packets = 0
pending = 0
total_pred = 0

# СТАТИСТИКА ЗАДЕРЖЕК ПО ЭТАПАМ

total_feature_time = 0.0
total_classifier_time = 0.0
total_scaler_time = 0.0
total_ordinal_time = 0.0
total_regression_time = 0.0
total_fusion_time = 0.0
total_all_time = 0.0

# PYGAME

pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("REAL-TIME DETECTION [0.45ms]")
clock = pygame.time.Clock()

font_l = pygame.font.SysFont("verdana", 28, bold=True)
font_m = pygame.font.SysFont("verdana", 22, bold=True)
font_ms = pygame.font.SysFont("verdana", 20, bold=True)
font_s = pygame.font.SysFont("verdana", 14)

last_time = time.perf_counter()
interval = 1.0 / SIMULATION_HZ

lower_risk_limit = 15
upper_risk_limit = 30

# ОСНОВНОЙ ЦИКЛ 

running = True
flad = True

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    
    now = time.perf_counter()
    
    while (now - last_time) >= interval and running:
        msg = MockMsg('HIGHRES_IMU')
        
        csv_buf.append([msg.time_usec, msg.xacc, msg.yacc, msg.zacc,
                       msg.xgyro, msg.ygyro, msg.zgyro, 0, 0, 0, 0, 0, 0, 25, 0, packets, '', '', '', ''])
        
        imu_buffer[:-1] = imu_buffer[1:]
        imu_buffer[-1] = [msg.xgyro, msg.ygyro, msg.zgyro, msg.xacc, msg.yacc, msg.zacc]
        
        packets += 1
        pending += 1
        
        if packets >= WINDOW_SIZE and pending >= STRIDE:
            
            # 1. Features extraction
            t_start = time.perf_counter()
            extract_features(imu_buffer, X_raw)
            X_sel[0, :] = X_raw[feature_indices]
            t_feature = time.perf_counter() - t_start
            
            # 2. Classifier inference
            t0 = time.perf_counter()
            current_class = int(sess_clf.run(None, {"input": X_sel})[0][0])
            t_classifier = time.perf_counter() - t0
            
            # 3. Scaling
            t0 = time.perf_counter()
            np.subtract(X_sel, mean, out=X_scaled)
            np.multiply(X_scaled, inv_scale, out=X_scaled)
            t_scaler = time.perf_counter() - t0
            
            # 4. Ordinal inference
            t0 = time.perf_counter()
            ordinal = (np.dot(X_scaled, W) + b).mean()
            ord_n = (ordinal - l_min) / denom_l
            t_ordinal = time.perf_counter() - t0
            
            # 5. Regression inference
            t0 = time.perf_counter()
            reg = sess_reg.run(None, {"input": X_sel})[0][0][0]
            reg_n = (reg - r_min) / denom_r
            t_regression = time.perf_counter() - t0
            
            # 6. Fusion
            t0 = time.perf_counter()
            current_risk = min(max(0, 100 * (alpha * ord_n + one_minus_alpha * reg_n)), 100)
            t_end = time.perf_counter()
            t_fusion = t_end - t0
            
            # Суммируем задержки
            # Сбор статистики после первых 10 предсказаний (прогрев)
            if total_pred > 1: 
                total_feature_time += t_feature
                total_classifier_time += t_classifier
                total_scaler_time += t_scaler
                total_ordinal_time += t_ordinal
                total_regression_time += t_regression
                total_fusion_time += t_fusion
                total_all_time += (t_end - t_start)
                current_latency = (t_end - t_start) * 1000
                
            total_pred += 1
                
            risk_history.append(float(current_risk))
            latency_history.append(float(current_latency))
            
            pending = 0
        
        last_time += interval
    
    # CSV batch
    if len(csv_buf) >= CSV_BATCH_SIZE:
        writer.writerows(csv_buf)
        csv_buf.clear()
    
    # ОТРИСОВКА 
    screen.fill((15, 15, 25))
    color = CLASS_COLORS[current_class] if 0 <= current_class <= 5 else (200, 200, 200)
    
    # Класс
    pygame.draw.rect(screen, (40, 40, 60), (20, 20, 545, 100), border_radius=10)
    pygame.draw.rect(screen, color, (20, 20, 545, 100), width=3, border_radius=10)
    screen.blit(font_s.render("ТЕКУЩЕЕ СОСТОЯНИЕ:", True, (220, 220, 220)), (40, 35))
    text = CLASS_LABELS.get(current_class, f"КЛАСС {current_class}")
    screen.blit(font_ms.render(text, True, color), (40, 65))
    
    # Задержка
    pygame.draw.rect(screen, (40, 40, 60), (WIDTH-320, 20, 300, 100), border_radius=10)
    screen.blit(font_s.render("ЗАДЕРЖКА", True, (220, 220, 220)), (WIDTH - 300, 40))
    screen.blit(font_m.render(f"{current_latency:.3f} ms", True, (220, 220, 220)), (WIDTH - 300, 65))
    
    # График задержки
    rect = pygame.Rect(WIDTH-160, 35, 120, 70)
    pygame.draw.rect(screen, (20, 20, 30), rect, border_radius=5)
    max_lat = max(max(latency_history), 1.0)
    lat_list = list(latency_history)
    
    for i in range(1, len(lat_list)):
        x1 = rect.x + int((i - 1) * rect.width / HISTORY_LEN)
        y1 = rect.bottom - int((lat_list[i - 1]/max_lat) * rect.height)
        x2 = rect.x + int(i * rect.width / HISTORY_LEN)
        y2 = rect.bottom - int((lat_list[i] / max_lat) * rect.height)
        pygame.draw.line(screen, (100, 200, 255), (x1, y1), (x2, y2), 2)
    
    # Статус
    screen.blit(font_s.render(f"Файл: {os.path.basename(file_path)}", True, (150, 150, 150)), (WIDTH - 318, 128))
    screen.blit(font_s.render(f"Записано: {packets}", True, (0, 255, 150)), (WIDTH - 318, 176))
    screen.blit(font_s.render(f"Предсказаний: {total_pred}", True, (0, 255, 150)), (WIDTH - 318, 152))

    
    # Риск
    g_rect = pygame.Rect(20, 200, WIDTH - 40, HEIGHT - 220)
    pygame.draw.rect(screen, (20, 20, 30), g_rect, border_radius=5)
    
    for p, col in [(lower_risk_limit, (0, 200, 100)), (upper_risk_limit, (255, 170, 0))]:
        y = g_rect.bottom - int((p / 100) * g_rect.height)
        pygame.draw.line(screen, col, (g_rect.left, y), (g_rect.right, y), 1)
        screen.blit(font_s.render(f"{p}%", True, col), (g_rect.left + 5, y - 20))
    
    if len(risk_history) > 1:
        risk_list = list(risk_history)
        pts = [(g_rect.left + int(i * g_rect.width / (HISTORY_LEN - 1)),
                g_rect.bottom - int((v / 100) * g_rect.height)) for i, v in enumerate(risk_list)]
        s_color = (0, 200, 100) if current_risk < lower_risk_limit else ((255, 170, 0) if current_risk < upper_risk_limit else (255, 50, 50))
        if len(pts) > 1: pygame.draw.lines(screen, s_color, False, pts, 3)
    
    screen.blit(font_l.render(f"Риск: {current_risk:.1f}%", True, s_color), (g_rect.left + 20, g_rect.top + 20))
    
    pygame.display.flip()
    clock.tick(FPS)

# ФИНАЛ 
if csv_buf:
    writer.writerows(csv_buf)
csv_file.close()
pygame.quit()

print("="*48)
print(" "*14, "ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ", sep="")
print("="*48)

label_width = 34

if total_pred > 0:
    avg_feature = (total_feature_time / total_pred) * 1000
    avg_classifier = (total_classifier_time / total_pred) * 1000
    avg_scaler = (total_scaler_time / total_pred) * 1000
    avg_ordinal = (total_ordinal_time / total_pred) * 1000
    avg_regression = (total_regression_time / total_pred) * 1000
    avg_fusion = (total_fusion_time / total_pred) * 1000

    avg_risk = avg_scaler + avg_ordinal + avg_regression + avg_fusion
    avg_all = avg_feature + avg_classifier + avg_risk

    print(f"\n{'Суммарная задержка обработки:':<{label_width}}{avg_all:.3f} ms\n")
    print(f"{'Суммарная задержка оценки риска:':<{label_width}}{avg_risk:.3f} ms\n")
    print(f"{'Задержка извлечения признаков:':<{label_width}}{avg_feature:.3f} ms")
    print(f"{'Задержка классификации:':<{label_width}}{avg_classifier:.3f} ms")
    print(f"{'Задержка масштабирования:':<{label_width}}{avg_scaler:.3f} ms")
    print(f"{'Задержка порядковой модели:':<{label_width}}{avg_ordinal:.3f} ms")
    print(f"{'Задержка регрессионной модели:':<{label_width}}{avg_regression:.3f} ms")

    
    
    print(f"\n{'Количество предсказаний:':<{label_width}}{total_pred}")
    print(f"{'Пропускная способность:':<{label_width}}{1000/avg_all:.1f} пред/с")

else:
    print("\nНет предсказаний")

print("\n", "="*48, sep="")


              ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ

Суммарная задержка обработки:     0.349 ms

Суммарная задержка оценки риска:  0.129 ms

Задержка извлечения признаков:    0.027 ms
Задержка классификации:           0.193 ms
Задержка масштабирования:         0.015 ms
Задержка порядковой модели:       0.045 ms
Задержка регрессионной модели:    0.065 ms

Количество предсказаний:          30
Пропускная способность:           2865.5 пред/с

